In [126]:
import numpy as np
import pandas as pd
from pathlib import Path

from sklearn.metrics import brier_score_loss, roc_auc_score

In [127]:
from llm_unsupervised_conf.metrics import get_ece1, get_ece2, get_mce, get_nll
from llm_unsupervised_conf.plots import DATASETS_MAP

In [128]:
score_cols = [
    "base_response_avg_logprob",
    "base_response_avg_prob",
    "base_response_num_tokens",
    "base_answer_avg_logprob",
    "base_answer_avg_prob",
    "base_answer_num_tokens",
]

comp_methods = [
    "base_response_avg_prob",
    "base_answer_avg_prob",
    "consistency"
]

base_dir = Path("../outputs")

datasets = [
    "gsm8k",
    "polymath",
    "sciq",
    "trivia_qa",
    "webq",
]

models = [
    "Qwen3-0.6B",
    "Qwen3-1.7B",
    "Qwen3-4B-Thinking-2507",
    "Qwen3-8B",
    "Qwen3-14B",
]

In [129]:
def metric_row(name, scores, correct, n_bins):
    return [
        name,
        get_ece1(scores, correct, n_bins=n_bins),
        get_ece2(scores, correct, n_bins=n_bins),
        get_mce(scores, correct, n_bins=n_bins),
        get_nll(scores, correct),
        brier_score_loss(correct, scores),
        roc_auc_score(correct, scores),
    ]

In [116]:
full_df = []

rows = []
for model in models:
    for dataset in datasets:
        csv_path = base_dir / model / dataset / f"{model.lower().replace('-', '_')}_{dataset}_with_base_targets.csv"

        # If your filenames are different, replace the line above with the exact pattern.
        if not csv_path.exists():
            rows.append({
                "model": model,
                "dataset": dataset,
                "exists": False,
                "all_scores_ok": False,
                "num_rows": np.nan,
                "num_bad_rows": np.nan,
            })
            continue

        df = pd.read_csv(csv_path)
        # print(f"No. rows before {len(df)}")
        df = df[df["answer"].notna() & (df["answer"].str.strip() != "") & (df["answer"] != "")]
        # print(f"No. rows after {len(df)}")
        # display(df.head())

        missing_cols = [c for c in score_cols if c not in df.columns]
        if missing_cols:
            rows.append({
                "model": model,
                "dataset": dataset,
                "exists": True,
                "all_scores_ok": False,
                "num_rows": len(df),
                "num_bad_rows": np.nan,
                "missing_cols": missing_cols,
            })
            continue

        vals = df[score_cols].to_numpy()
        row_ok = df[score_cols].notna().all(axis=1) & np.isfinite(vals).all(axis=1)

        rows.append({
            "model": model,
            "dataset": dataset,
            "exists": True,
            "all_scores_ok": bool(row_ok.all()),
            "num_rows": len(df),
            "num_bad_rows": int((~row_ok).sum()),
            "missing_cols": None,
        })

        results = [metric_row(m, df[m], df["correct"], n_bins=12) for m in comp_methods]
        results_df = pd.DataFrame(results, columns=["method", "ECE1", "ECE2", "MCE", "NLL", "Brier", "AUROC"])
        

        results_df["model"] = model.split("/")[-1]
        results_df["dataset"] = dataset

        results_df = results_df[["model", "dataset", "method", "ECE1", "ECE2", "MCE", "NLL", "Brier", "AUROC"]]

        display(results_df)

        full_df.append(results_df)

full_df = pd.concat(full_df)

summary_df = pd.DataFrame(rows)
summary_df

,model,dataset,method,ECE1,ECE2,MCE,NLL,Brier,AUROC
0,Qwen3-0.6B,gsm8k,base_response_avg_prob,0.346541,0.376866,0.520339,0.767939,0.286903,0.331478
1,Qwen3-0.6B,gsm8k,base_answer_avg_prob,0.690378,0.706161,0.855786,2.263272,0.651136,0.495403
2,Qwen3-0.6B,gsm8k,consistency,0.055110,0.066317,0.132619,0.422315,0.106006,0.865107


,model,dataset,method,ECE1,ECE2,MCE,NLL,Brier,AUROC
0,Qwen3-0.6B,polymath,base_response_avg_prob,0.277464,0.308411,0.479430,0.777835,0.291402,0.302196
1,Qwen3-0.6B,polymath,base_answer_avg_prob,0.471811,0.501558,0.649662,1.493432,0.482037,0.484775
2,Qwen3-0.6B,polymath,consistency,0.106690,0.123699,0.241852,0.545494,0.160788,0.854580


,model,dataset,method,ECE1,ECE2,MCE,NLL,Brier,AUROC
0,Qwen3-0.6B,sciq,base_response_avg_prob,0.127479,0.138417,0.224376,0.716598,0.258857,0.466998
1,Qwen3-0.6B,sciq,base_answer_avg_prob,0.394750,0.415609,0.543154,3.218187,0.403542,0.413862
2,Qwen3-0.6B,sciq,consistency,0.060897,0.078820,0.171154,0.592622,0.203291,0.744757


,model,dataset,method,ECE1,ECE2,MCE,NLL,Brier,AUROC
0,Qwen3-0.6B,trivia_qa,base_response_avg_prob,0.112392,0.145255,0.332861,0.567971,0.189343,0.422325
1,Qwen3-0.6B,trivia_qa,base_answer_avg_prob,0.194288,0.203883,0.259227,1.734038,0.207786,0.493898
2,Qwen3-0.6B,trivia_qa,consistency,0.056389,0.061437,0.091013,0.437530,0.139708,0.787908


,model,dataset,method,ECE1,ECE2,MCE,NLL,Brier,AUROC
0,Qwen3-0.6B,webq,base_response_avg_prob,0.103460,0.135617,0.353469,0.546528,0.179585,0.427120
1,Qwen3-0.6B,webq,base_answer_avg_prob,0.213646,0.221052,0.353909,1.496717,0.213449,0.467495
2,Qwen3-0.6B,webq,consistency,0.115155,0.149379,0.284318,0.542837,0.180901,0.640037


...

,model,dataset,method,ECE1,ECE2,MCE,NLL,Brier,AUROC
0,Qwen3-14B,polymath,base_response_avg_prob,0.369771,0.376797,0.506570,0.623917,0.215787,0.432949
1,Qwen3-14B,polymath,base_answer_avg_prob,0.597962,0.635269,0.841524,1.403445,0.476245,0.662531
2,Qwen3-14B,polymath,consistency,0.051436,0.104649,0.299425,0.273727,0.055983,0.911644


,model,dataset,method,ECE1,ECE2,MCE,NLL,Brier,AUROC
0,Qwen3-14B,sciq,base_response_avg_prob,0.232768,0.254465,0.367199,0.741931,0.274123,0.424630
1,Qwen3-14B,sciq,base_answer_avg_prob,0.671711,0.693788,0.781668,4.147316,0.679176,0.362804
2,Qwen3-14B,sciq,consistency,0.109240,0.129105,0.261446,0.753745,0.195296,0.716474


,model,dataset,method,ECE1,ECE2,MCE,NLL,Brier,AUROC
0,Qwen3-14B,trivia_qa,base_response_avg_prob,0.260876,0.276025,0.381060,0.759512,0.282715,0.449760
1,Qwen3-14B,trivia_qa,base_answer_avg_prob,0.654186,0.680808,0.817899,4.183697,0.658587,0.482168
2,Qwen3-14B,trivia_qa,consistency,0.031992,0.046125,0.109333,0.415131,0.129973,0.871588


,model,dataset,method,ECE1,ECE2,MCE,NLL,Brier,AUROC
0,Qwen3-14B,webq,base_response_avg_prob,0.106758,0.139462,0.306554,0.710574,0.258507,0.411058
1,Qwen3-14B,webq,base_answer_avg_prob,0.388632,0.416866,0.584313,2.137286,0.422300,0.465877
2,Qwen3-14B,webq,consistency,0.149667,0.170950,0.302429,0.852120,0.248258,0.687650


,model,dataset,exists,all_scores_ok,num_rows,num_bad_rows,missing_cols
0,Qwen3-0.6B,gsm8k,True,True,998,0,None
1,Qwen3-0.6B,polymath,True,True,997,0,None
2,Qwen3-0.6B,sciq,True,True,970,0,None
3,Qwen3-0.6B,trivia_qa,True,True,961,0,None
4,Qwen3-0.6B,webq,True,True,582,0,None
5,Qwen3-1.7B,gsm8k,True,True,999,0,None
6,Qwen3-1.7B,polymath,True,True,1000,0,None
7,Qwen3-1.7B,sciq,True,True,1000,0,None
8,Qwen3-1.7B,trivia_qa,True,True,996,0,None
9,Qwen3-1.7B,webq,True,True,986,0,None


In [107]:
full_df

,model,dataset,method,ECE1,ECE2,MCE,NLL,Brier,AUROC
0,Qwen3-0.6B,gsm8k,base_response_avg_prob,0.346541,0.376866,0.520339,0.767939,0.286903,0.331478
1,Qwen3-0.6B,gsm8k,base_answer_avg_prob,0.690378,0.706161,0.855786,2.263272,0.651136,0.495403
2,Qwen3-0.6B,gsm8k,consistency,0.055110,0.066317,0.132619,0.422315,0.106006,0.865107
0,Qwen3-0.6B,polymath,base_response_avg_prob,0.277464,0.308411,0.479430,0.777835,0.291402,0.302196
1,Qwen3-0.6B,polymath,base_answer_avg_prob,0.471811,0.501558,0.649662,1.493432,0.482037,0.484775
...,...,...,...,...,...,...,...,...,...
1,Qwen3-14B,trivia_qa,base_answer_avg_prob,0.654186,0.680808,0.817899,4.183697,0.658587,0.482168
2,Qwen3-14B,trivia_qa,consistency,0.031992,0.046125,0.109333,0.415131,0.129973,0.871588
0,Qwen3-14B,webq,base_response_avg_prob,0.106758,0.139462,0.306554,0.710574,0.258507,0.411058
1,Qwen3-14B,webq,base_answer_avg_prob,0.388632,0.416866,0.584313,2.137286,0.422300,0.465877


In [108]:
show_cols = ["method", "ECE2", "Brier", "AUROC"]
labels_map = {
    "base_answer_avg_prob": "Base Answer Probs.",
    "base_response_avg_prob": "Base Response Probs.",
    "consistency": "Self-consistency"
}


In [109]:
model_df = full_df[["model"]+show_cols].groupby(["model", "method"]).mean().reset_index()
model_df["method"] = [labels_map[m] for m in model_df["method"].tolist()]
model_df

,model,method,ECE2,Brier,AUROC
0,Qwen3-0.6B,Base Answer Probs.,0.409653,0.391590,0.471087
1,Qwen3-0.6B,Base Response Probs.,0.220913,0.241218,0.390024
2,Qwen3-0.6B,Self-consistency,0.095931,0.158139,0.778478
3,Qwen3-1.7B,Base Answer Probs.,0.567615,0.543247,0.458060
4,Qwen3-1.7B,Base Response Probs.,0.272661,0.256348,0.343952
5,Qwen3-1.7B,Self-consistency,0.107435,0.153393,0.779973
6,Qwen3-14B,Base Answer Probs.,0.638084,0.573921,0.507222
7,Qwen3-14B,Base Response Probs.,0.291145,0.249500,0.433789
8,Qwen3-14B,Self-consistency,0.100954,0.134540,0.787142
9,Qwen3-4B-Thinking-2507,Base Answer Probs.,0.635445,0.594852,0.496000


In [131]:
print(model_df.to_latex(index=False, float_format="%.3f"))

\begin{tabular}{llrrr}
\toprule
model & method & ECE2 & Brier & AUROC \\
\midrule
Qwen3-0.6B & Base Answer Probs. & 0.410 & 0.392 & 0.471 \\
Qwen3-0.6B & Base Response Probs. & 0.221 & 0.241 & 0.390 \\
Qwen3-0.6B & Self-consistency & 0.096 & 0.158 & 0.778 \\
Qwen3-1.7B & Base Answer Probs. & 0.568 & 0.543 & 0.458 \\
Qwen3-1.7B & Base Response Probs. & 0.273 & 0.256 & 0.344 \\
Qwen3-1.7B & Self-consistency & 0.107 & 0.153 & 0.780 \\
Qwen3-14B & Base Answer Probs. & 0.638 & 0.574 & 0.507 \\
Qwen3-14B & Base Response Probs. & 0.291 & 0.250 & 0.434 \\
Qwen3-14B & Self-consistency & 0.101 & 0.135 & 0.787 \\
Qwen3-4B-Thinking-2507 & Base Answer Probs. & 0.635 & 0.595 & 0.496 \\
Qwen3-4B-Thinking-2507 & Base Response Probs. & 0.325 & 0.271 & 0.410 \\
Qwen3-4B-Thinking-2507 & Self-consistency & 0.089 & 0.137 & 0.777 \\
Qwen3-8B & Base Answer Probs. & 0.664 & 0.626 & 0.482 \\
Qwen3-8B & Base Response Probs. & 0.280 & 0.244 & 0.366 \\
Qwen3-8B & Self-consistency & 0.098 & 0.140 & 0.777 \\
\botto

In [121]:
dataset_df = full_df[["dataset"]+show_cols].groupby(["dataset", "method"]).mean().reset_index()
dataset_df["method"] = [labels_map[m] for m in dataset_df["method"].tolist()]
dataset_df["dataset"] = [DATASETS_MAP[d] for d in dataset_df["dataset"].tolist()]
dataset_df

,dataset,method,ECE2,Brier,AUROC
0,GSM8K,Base Answer Probs.,0.806625,0.732395,0.490491
1,GSM8K,Base Response Probs.,0.401105,0.235144,0.385330
2,GSM8K,Self-consistency,0.055884,0.059957,0.781150
3,Polymath,Base Answer Probs.,0.674811,0.580987,0.543555
4,Polymath,Base Response Probs.,0.362323,0.237269,0.353366
5,Polymath,Self-consistency,0.103993,0.083712,0.887981
6,SciQ,Base Answer Probs.,0.607929,0.594712,0.396901
7,SciQ,Base Response Probs.,0.245220,0.286563,0.418328
8,SciQ,Self-consistency,0.103121,0.199850,0.728588
9,Trivia QA,Base Answer Probs.,0.478923,0.469357,0.504149


In [132]:
print(dataset_df.to_latex(index=False, float_format="%.3f"))

\begin{tabular}{llrrr}
\toprule
dataset & method & ECE2 & Brier & AUROC \\
\midrule
GSM8K & Base Answer Probs. & 0.807 & 0.732 & 0.490 \\
GSM8K & Base Response Probs. & 0.401 & 0.235 & 0.385 \\
GSM8K & Self-consistency & 0.056 & 0.060 & 0.781 \\
Polymath & Base Answer Probs. & 0.675 & 0.581 & 0.544 \\
Polymath & Base Response Probs. & 0.362 & 0.237 & 0.353 \\
Polymath & Self-consistency & 0.104 & 0.084 & 0.888 \\
SciQ & Base Answer Probs. & 0.608 & 0.595 & 0.397 \\
SciQ & Base Response Probs. & 0.245 & 0.287 & 0.418 \\
SciQ & Self-consistency & 0.103 & 0.200 & 0.729 \\
Trivia QA & Base Answer Probs. & 0.479 & 0.469 & 0.504 \\
Trivia QA & Base Response Probs. & 0.245 & 0.263 & 0.357 \\
Trivia QA & Self-consistency & 0.063 & 0.149 & 0.840 \\
WebQ & Base Answer Probs. & 0.346 & 0.352 & 0.480 \\
WebQ & Base Response Probs. & 0.136 & 0.240 & 0.429 \\
WebQ & Self-consistency & 0.166 & 0.231 & 0.662 \\
\bottomrule
\end{tabular}

